# Chương 4. Hồi quy tuyến tính (Linear Regression)
*Giáo trình **Học máy – Cơ sở và Ứng dụng với Python***

Notebook thực hành đi kèm Chương 4. Chạy lần lượt từng ô (**Runtime → Run all**). Không cần GPU, không cần tải dữ liệu ngoài (chỉ dùng dữ liệu có sẵn trong `scikit-learn` và dữ liệu giả lập).

## Mục tiêu học tập
Sau bài này, bạn có thể:
1. Viết mô hình hồi quy tuyến tính đơn và đa biến, hiểu ý nghĩa của các hệ số.
2. Giải bằng **phương trình chuẩn (OLS)** và bằng **Gradient Descent** tự cài đặt.
3. Phân tích ảnh hưởng của **tốc độ học** $\alpha$.
4. Đánh giá mô hình bằng MAE, MSE, RMSE, $R^2$ và kiểm tra **phần dư**.
5. Nhận biết các giả định của mô hình và hiện tượng đa cộng tuyến.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.rcParams["figure.figsize"] = (7, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
np.random.seed(42)
print("Đã sẵn sàng!")

## 1. Mô hình hồi quy tuyến tính

**Đơn biến:** $\hat y = w_0 + w_1 x$

**Đa biến:** $\hat y = w_0 + w_1 x_1 + \dots + w_d x_d = \mathbf{w}^\top \mathbf{x}$ (với $x_0 = 1$)

- $w_0$: hệ số chặn (intercept) — giá trị dự đoán khi mọi $x_j=0$.
- $w_j$: khi $x_j$ tăng 1 đơn vị (các biến khác giữ nguyên), $\hat y$ thay đổi $w_j$ đơn vị.

**Hàm mất mát (MSE):**
$$J(\mathbf{w}) = \frac{1}{n}\sum_{i=1}^{n}\left(y_i - \hat y_i\right)^2$$

**Nghiệm đóng (phương trình chuẩn, OLS):**
$$\mathbf{w}^* = (X^\top X)^{-1} X^\top \mathbf{y}$$

## 2. Ví dụ: dự đoán mức độ tiến triển bệnh tiểu đường từ chỉ số BMI

Bộ dữ liệu `load_diabetes` có 442 bệnh nhân, 10 đặc trưng (đã chuẩn hoá) và biến mục tiêu là chỉ số tiến triển bệnh sau 1 năm. Trước hết chỉ dùng **một** đặc trưng: `bmi`.

In [ ]:
du_lieu = load_diabetes()
print("Kích thước X:", du_lieu.data.shape)
print("Các đặc trưng:", du_lieu.feature_names)

df = pd.DataFrame(du_lieu.data, columns=du_lieu.feature_names)
df["muc_tieu"] = du_lieu.target
df.head()

In [ ]:
chi_so_bmi = du_lieu.feature_names.index("bmi")
X = du_lieu.data[:, [chi_so_bmi]]
y = du_lieu.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

mo_hinh = LinearRegression().fit(X_train, y_train)
y_du_doan = mo_hinh.predict(X_test)

print(f"w0 (hệ số chặn) = {mo_hinh.intercept_:.2f}")
print(f"w1 (hệ số BMI)  = {mo_hinh.coef_[0]:.2f}")

In [ ]:
mae = mean_absolute_error(y_test, y_du_doan)
mse = mean_squared_error(y_test, y_du_doan)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_du_doan)

print(f"MAE  = {mae:.2f}")
print(f"MSE  = {mse:.2f}")
print(f"RMSE = {rmse:.2f}")
print(f"R²   = {r2:.4f}")

**Giải thích kết quả.**
- Sai số tuyệt đối trung bình khoảng 53 đơn vị so với thang mục tiêu (25–346).
- $R^2 \approx 0{,}23$: chỉ riêng BMI giải thích được khoảng 23% phương sai của biến mục tiêu — mô hình một biến còn khá đơn giản.

In [ ]:
plt.scatter(X_test, y_test, alpha=0.7, label="Dữ liệu kiểm tra")
xs = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
plt.plot(xs, mo_hinh.predict(xs), color="crimson", lw=2, label="Đường hồi quy")
plt.xlabel("BMI (đã chuẩn hoá)")
plt.ylabel("Chỉ số tiến triển bệnh")
plt.title("Hồi quy tuyến tính đơn biến")
plt.legend()
plt.show()

## 3. Giải bằng phương trình chuẩn (tự cài đặt)

Kiểm tra rằng công thức $\mathbf{w}^* = (X^\top X)^{-1}X^\top \mathbf{y}$ cho kết quả trùng với `LinearRegression`.

In [ ]:
def ols(X, y):
    Xb = np.c_[np.ones(len(X)), X]          # thêm cột 1 cho w0
    return np.linalg.solve(Xb.T @ Xb, Xb.T @ y)   # ổn định hơn nghịch đảo trực tiếp

w = ols(X_train, y_train)
print("OLS tự cài  :", np.round(w, 2))
print("scikit-learn:", np.round([mo_hinh.intercept_, mo_hinh.coef_[0]], 2))

## 4. Gradient Descent

Khi dữ liệu lớn, ta cập nhật lặp:
$$\mathbf{w} \leftarrow \mathbf{w} - \alpha \,\nabla_{\mathbf{w}} J, \qquad \nabla_{\mathbf{w}} J = -\frac{2}{n}X^\top(\mathbf{y}-X\mathbf{w})$$

Ta thử trên dữ liệu giả lập $y = 3x + 2 + \varepsilon$.

In [ ]:
rng = np.random.default_rng(0)
x = rng.uniform(-3, 3, 100)
y_gl = 3 * x + 2 + rng.normal(0, 1.0, 100)

def gradient_descent(x, y, alpha=0.05, so_lan=100):
    n = len(x)
    w0, w1 = 0.0, 0.0
    lich_su = []
    for _ in range(so_lan):
        y_hat = w0 + w1 * x
        loi = y - y_hat
        lich_su.append(np.mean(loi ** 2))
        w0 += alpha * 2 / n * loi.sum()
        w1 += alpha * 2 / n * (loi * x).sum()
    return w0, w1, lich_su

w0, w1, mat_mat = gradient_descent(x, y_gl)
print(f"Gradient Descent: w0 = {w0:.3f}, w1 = {w1:.3f}  (giá trị thật: 2 và 3)")
print("OLS            :", np.round(ols(x.reshape(-1, 1), y_gl), 3))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(mat_mat, lw=2)
ax[0].set_xlabel("Số vòng lặp"); ax[0].set_ylabel("MSE")
ax[0].set_title("Đường cong mất mát")

ax[1].scatter(x, y_gl, alpha=0.6, label="Dữ liệu")
xs = np.linspace(-3, 3, 100)
ax[1].plot(xs, w0 + w1 * xs, color="crimson", lw=2, label="GD")
ax[1].set_title("Đường thẳng sau khi học"); ax[1].legend()
plt.tight_layout(); plt.show()

### Ảnh hưởng của tốc độ học $\alpha$
- $\alpha$ quá nhỏ → hội tụ rất chậm.
- $\alpha$ vừa phải → hội tụ nhanh và ổn định.
- $\alpha$ quá lớn → dao động hoặc **phân kỳ**.

In [ ]:
for alpha in [0.001, 0.01, 0.05, 0.3]:
    _, _, ls = gradient_descent(x, y_gl, alpha=alpha, so_lan=60)
    plt.plot(ls, label=f"α = {alpha}")
plt.yscale("log")
plt.xlabel("Số vòng lặp"); plt.ylabel("MSE (thang log)")
plt.title("So sánh các tốc độ học"); plt.legend(); plt.show()

In [ ]:
# Thử một giá trị quá lớn để thấy phân kỳ
_, _, ls = gradient_descent(x, y_gl, alpha=0.5, so_lan=30)
print("MSE sau 30 vòng với α = 0.5:", f"{ls[-1]:.3e}", "→ phân kỳ!" if ls[-1] > ls[0] else "")

## 5. Hồi quy đa biến với toàn bộ 10 đặc trưng

In [ ]:
Xa = du_lieu.data
Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(Xa, y, test_size=0.2, random_state=42)

mh_da = LinearRegression().fit(Xa_tr, ya_tr)
dd = mh_da.predict(Xa_te)

print(f"MAE  = {mean_absolute_error(ya_te, dd):.2f}")
print(f"RMSE = {np.sqrt(mean_squared_error(ya_te, dd)):.2f}")
print(f"R²   = {r2_score(ya_te, dd):.4f}")

he_so = pd.Series(mh_da.coef_, index=du_lieu.feature_names).sort_values()
he_so.plot.barh(color="steelblue", title="Hệ số của mô hình đa biến")
plt.xlabel("Giá trị hệ số"); plt.show()

So với mô hình chỉ có BMI, $R^2$ tăng rõ rệt — thêm thông tin hợp lý giúp mô hình giải thích dữ liệu tốt hơn.

**Lưu ý đa cộng tuyến:** các đặc trưng `s1`…`s6` (xét nghiệm máu) tương quan mạnh với nhau, khiến hệ số có thể lớn, trái dấu và kém ổn định. Kiểm tra bằng ma trận tương quan:

In [ ]:
import matplotlib as mpl
tuong_quan = df.drop(columns="muc_tieu").corr()
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(tuong_quan, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(10)); ax.set_xticklabels(tuong_quan.columns, rotation=45)
ax.set_yticks(range(10)); ax.set_yticklabels(tuong_quan.columns)
ax.grid(False)
for i in range(10):
    for j in range(10):
        ax.text(j, i, f"{tuong_quan.iloc[i, j]:.1f}", ha="center", va="center", fontsize=7)
fig.colorbar(im); ax.set_title("Ma trận tương quan giữa các đặc trưng")
plt.tight_layout(); plt.show()

## 6. Kiểm tra giả định qua phần dư

Phần dư $e_i = y_i - \hat y_i$. Nếu mô hình phù hợp, phần dư nên: phân bố quanh 0, không có cấu trúc rõ theo $\hat y$ (đồng phương sai), và xấp xỉ chuẩn.

In [ ]:
phan_du = ya_te - dd
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))

ax[0].scatter(dd, phan_du, alpha=0.7)
ax[0].axhline(0, color="crimson", ls="--")
ax[0].set_xlabel("Giá trị dự đoán"); ax[0].set_ylabel("Phần dư")
ax[0].set_title("Phần dư theo giá trị dự đoán")

ax[1].hist(phan_du, bins=20, color="steelblue", edgecolor="white")
ax[1].set_xlabel("Phần dư"); ax[1].set_title("Phân bố phần dư")
plt.tight_layout(); plt.show()

## 7. Bài tập

### Mức 1 – Củng cố
1. Đổi đặc trưng từ `bmi` sang `bp` (huyết áp) ở Mục 2. $R^2$ tăng hay giảm? Đặc trưng đơn lẻ nào cho $R^2$ cao nhất?
2. Thay `test_size=0.2` bằng 0.3 và `random_state` khác. Kết quả thay đổi thế nào? Vì sao?

### Mức 2 – Vận dụng
3. Sửa hàm `gradient_descent` để dừng sớm khi MSE giảm ít hơn $10^{-6}$ giữa hai vòng lặp.
4. Cài đặt Gradient Descent **đa biến** dạng ma trận và so sánh với `LinearRegression` trên 10 đặc trưng.

### Mức 3 – Nâng cao
5. Chuẩn hoá đặc trưng bằng `StandardScaler` rồi so sánh tốc độ hội tụ của GD trước và sau chuẩn hoá.
6. Sinh dữ liệu có quan hệ phi tuyến $y = x^2 + \varepsilon$, hồi quy tuyến tính và quan sát phần dư. Sau đó thêm đặc trưng $x^2$ (`PolynomialFeatures`) và so sánh.

### Bài tập lập trình
7. Với bộ **California Housing** (`sklearn.datasets.fetch_california_housing`, cần Internet — chạy được trên Colab), huấn luyện hồi quy tuyến tính và vẽ biểu đồ phân tán *giá thực tế – giá dự đoán* kèm đường $y=x$.

In [ ]:
# Khu vực làm bài tập — viết code của bạn ở đây

## 8. Tóm tắt
- Hồi quy tuyến tính mô hình hoá $\hat y = \mathbf{w}^\top\mathbf{x}$, huấn luyện bằng cách cực tiểu MSE.
- Có thể giải bằng nghiệm đóng (OLS) hoặc Gradient Descent; $\alpha$ phải được chọn cẩn thận.
- Đánh giá bằng MAE, RMSE, $R^2$ trên **tập kiểm tra**; kiểm tra phần dư để phát hiện vi phạm giả định.
- Đa cộng tuyến làm hệ số kém ổn định (xem thêm Ridge/Lasso ở chương sau).